In [0]:
# Load medication data into a dataframe for reuse
df_medication = spark.sql("""
    SELECT *
    FROM _exponent._bronze_allscripts_tw_works.dbo_item_medication
    -- WHERE CreateDTTM >= '2024-01-01'
    LIMIT 100
""")

display(df_medication)

In [0]:
%sql DESCRIBE _exponent._bronze_allscripts_tw_works.dbo_item_medication

In [0]:
df_medication.printSchema()
# # Get the column names and data types
# columns = df_medication.schema.names
# data_types = [df_medication.schema[col].dataType for col in columns]
# # Create a new DataFrame with the column names and data types
# df_schema = spark.createDataFrame([(col, data_type) for col, data_type in zip(columns, data_types)], ['col_name', 'data_type'])
# display(df_schema)

In [0]:
df_medication.describe().display()

Databricks data profile. Run in Databricks to view.

In [0]:
%sql
-- dbo_item_medication: Maps to drug_exposure (not in current scope)
-- For now, only validating patient linkage to person table
-- Future: Full mapping to drug_exposure table will include:
--   - EntryName → drug_concept_id (via concept lookup)
--   - DecodedValue → sig (dosing instructions)
--   - PerformedDTTM/CreateDTTM → drug_exposure_start_date
--   - ActivityHeaderID → visit_occurrence_id (via encounter lookup)

SELECT DISTINCT
    PatientID AS person_id
FROM _exponent._bronze_allscripts_tw_works.dbo_item_medication
WHERE PatientID IS NOT NULL
LIMIT 100

In [0]:
%sql
-- WRITE TO OMOP: drug_exposure
-- Source: dbo_item_medication

SELECT
    ID AS drug_exposure_id,
    PatientID AS person_id,
    NULL AS drug_concept_id,  -- TODO: Join to concept table on EntryName or CommunityItemID
    CAST(COALESCE(PerformedDTTM, CreateDTTM) AS DATE) AS drug_exposure_start_date,
    CAST(COALESCE(PerformedDTTM, CreateDTTM) AS TIMESTAMP) AS drug_exposure_start_datetime,
    NULL AS drug_exposure_end_date,
    NULL AS drug_exposure_end_datetime,
    32817 AS drug_type_concept_id,  -- 'EHR prescription'
    NULL AS stop_reason,
    NULL AS refills,
    NULL AS quantity,
    NULL AS days_supply,
    DecodedValue AS sig,
    NULL AS route_concept_id,
    NULL AS lot_number,
    NULL AS provider_id,
    NULL AS visit_occurrence_id,  -- TODO: Link via ActivityHeaderID to encounter table
    NULL AS visit_detail_id,
    EntryName AS drug_source_value,
    NULL AS drug_source_concept_id,
    NULL AS route_source_value,
    NULL AS dose_unit_source_value
FROM _exponent._bronze_allscripts_tw_works.dbo_item_medication
WHERE PatientID IS NOT NULL
LIMIT 100

In [0]:
df_medication_dbo = spark.sql("""
    SELECT * 
    FROM _exponent._bronze_allscripts_tw_works.dbo_medication
    LIMIT 100
""")

df_medication_dbo.printSchema()

In [0]:
%sql
-- WRITE TO OMOP: drug_exposure
-- Source: dbo_medication
-- Note: This table has more detailed prescription data than dbo_item_medication

SELECT
    ID AS drug_exposure_id,
    WhoForID AS person_id,  -- Patient receiving medication
    NULL AS drug_concept_id,  -- TODO: Join to concept via NDC or MedDictDE
    CAST(COALESCE(StartFuzzySortAs, PerformedFuzzySortAs, RecordedDTTM) AS DATE) AS drug_exposure_start_date,
    COALESCE(StartFuzzySortAs, PerformedFuzzySortAs, RecordedDTTM) AS drug_exposure_start_datetime,
    CAST(EndFuzzySortAs AS DATE) AS drug_exposure_end_date,
    EndFuzzySortAs AS drug_exposure_end_datetime,
    32817 AS drug_type_concept_id,  -- 'EHR prescription'
    NULL AS stop_reason,  -- Could derive from MedicationStatusDE
    Refill AS refills,
    QuantityToDispense AS quantity,
    DaysSupply AS days_supply,
    COALESCE(FreeTextSIG, Instructions) AS sig,
    NULL AS route_concept_id,  -- TODO: Map RoutOfAdministrationDE or AdministrationRouteDE to concept
    AdministrationLot AS lot_number,
    PrescribedByID AS provider_id,
    NoteActivityID AS visit_occurrence_id,  -- TODO: Verify linkage
    NULL AS visit_detail_id,
    NDC AS drug_source_value,
    NULL AS drug_source_concept_id,  -- TODO: Map NDC to source concept
    RoutOfAdministrationDE AS route_source_value,
    Dose AS dose_unit_source_value
FROM _exponent._bronze_allscripts_tw_works.dbo_medication
WHERE WhoForID IS NOT NULL
LIMIT 100

In [0]:
df_medication_de = spark.sql("""
    SELECT * 
    FROM _exponent._bronze_allscripts_tw_works.dbo_medication_de
    LIMIT 100
""")

df_medication_de.printSchema()

This is a reference/lookup table for medications - not transactional data. It contains the drug dictionary with codes and attributes.
This maps to concept and drug_strength:

In [0]:
%sql
-- WRITE TO OMOP: concept (medication reference data)
-- Source: dbo_medication_de
-- Note: This is a drug dictionary/lookup table, not patient transactions

SELECT
    ID AS concept_id,  -- Or generate surrogate key
    DrugName AS concept_name,
    'Drug' AS domain_id,
    'RxNorm' AS vocabulary_id,  -- TODO: Verify based on source system
    Form AS concept_class_id,  -- e.g., Tablet, Capsule
    NULL AS standard_concept,
    RxNormCodeNormalized AS concept_code,
    EffectiveDT AS valid_start_date,
    COALESCE(KeepActiveUntil, CAST('2099-12-31' AS DATE)) AS valid_end_date,
    CASE WHEN IsInactiveFLAG = 'Y' THEN 'D' ELSE NULL END AS invalid_reason
FROM _exponent._bronze_allscripts_tw_works.dbo_medication_de
WHERE IsInactiveFLAG = 'N'
LIMIT 100

In [0]:
%sql
-- WRITE TO OMOP: drug_strength
-- Source: dbo_medication_de

SELECT
    ID AS drug_concept_id,  -- Links to concept above
    NULL AS ingredient_concept_id,  -- TODO: Parse from GenericName or lookup
    Strength AS amount_value,  -- May need to parse numeric value
    NULL AS amount_unit_concept_id,  -- TODO: Parse unit from Strength or UnitOfMeasure
    NULL AS numerator_value,
    NULL AS numerator_unit_concept_id,
    NULL AS denominator_value,
    NULL AS denominator_unit_concept_id,
    NULL AS box_size,
    EffectiveDT AS valid_start_date,
    COALESCE(KeepActiveUntil, CAST('2099-12-31' AS DATE)) AS valid_end_date,
    CASE WHEN IsInactiveFLAG = 'Y' THEN 'D' ELSE NULL END AS invalid_reason
FROM _exponent._bronze_allscripts_tw_works.dbo_medication_de
WHERE Strength IS NOT NULL
LIMIT 100

In [0]:
df_order_activity_header = spark.sql("""
    SELECT * 
    FROM _exponent._bronze_allscripts_tw_works.dbo_order_activity_header
    LIMIT 100
""")

df_order_activity_header.printSchema()
df_order_activity_header.display()

This is an activity/encounter linkage table. It connects orders to visits and encounters.
This doesn't map directly to one of your 8 OMOP tables, but it's a critical bridge table for populating visit_occurrence_id in drug_exposure:

In [0]:
%sql
-- REFERENCE TABLE: dbo_order_activity_header
-- Purpose: Links orders to visits/encounters
-- Maps to: None directly (bridge table)
-- Used by: drug_exposure.visit_occurrence_id lookups

-- Key relationships:
--   dbo_item_medication.ActivityHeaderID → dbo_order_activity_header.ID
--   dbo_order_activity_header.VisitID → visit_occurrence (future)
--   dbo_order_activity_header.EncounterID → visit_occurrence (future)
--   dbo_order_activity_header.PatientID → person.person_id

SELECT
    ID AS activity_header_id,
    PatientID AS person_id,
    VisitID AS visit_id,
    EncounterID AS encounter_id,
    CreateDTTM AS activity_datetime,
    ActivityType AS activity_type,
    ActivityTypeDE AS activity_type_de
FROM _exponent._bronze_allscripts_tw_works.dbo_order_activity_header
LIMIT 100

Updated drug_exposure Join
You can now update the dbo_item_medication transformation:

-- Update visit_occurrence_id in drug_exposure from dbo_item_medication
SELECT
    m.ID AS drug_exposure_id,
    m.PatientID AS person_id,
    ...
    oah.VisitID AS visit_occurrence_id,  -- Now populated via join
    ...
FROM _exponent._bronze_allscripts_tw_works.dbo_item_medication m
LEFT JOIN _exponent._bronze_allscripts_tw_works.dbo_order_activity_header oah
    ON m.ActivityHeaderID = oah.ID